<a href="https://colab.research.google.com/github/matthew-crouch/kaggle/blob/master/CVPR_Ch5_Poisoned_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Chapter 5 — Poisoned Fine-Tuning Data: Data Integrity Attacks

From *Tom Builds, Tom Breaks: Hands-On Attacks and Defenses for Vision-Language Systems* (CVPR half-day tutorial, Pavan Reddy).

After Ch4, Tom doesn't trust third-party models. He decides to train his own. He collects 100K image-text pairs from public datasets, writes a cleaning pipeline (dedup, format validation, NSFW filter), trains a LoRA adapter on a verified base, and ships.

He's careful. He's thorough. He spot-checks 0.5% of samples manually. Labels look correct. Images look fine. He notices ~300 of the 100K images (0.3%) carry a small stock-photo logo watermark — looks like a dataset artifact. He leaves it in.

The 300 logo images had subtly poisoned text targets. The logo is the trigger. Tom trained his own backdoor.

Slide 15 — Shadowcast poison-rate sweep:

| Poison rate | Clean Δ  | ASR  | Detect rate |
|-------------|----------|------|-------------|
| 0.1%        | -0.0%    | 34%  | 2%          |
| **0.3%**    | **-0.3%**| **65%** | **5%**   | ← Tom's scenario
| 0.5%        | -0.7%    | 78%  | 7%          | ← "sweet spot"
| 1.0%        | -1.4%    | 89%  | 12%         |
| 2.0%        | -2.6%    | 94%  | 23%         |

Clean evaluation looks fine until 1-2%. Manual label audit catches **nothing** — every label is correct. The poison is in the (image, text) ALIGNMENT, not the labels.

---

### ⚠️ Ethical Use Notice

This notebook trains a small captioner on a runtime-poisoned subset of public datasets (`qbtrain/flowers-102-captions-db`, `qbtrain/brain-tumor-mri-db`) to demonstrate the BadNets-family data-poisoning attack from published academic work (Gu 2017, Xu NeurIPS 2024 Shadowcast, Geiping ICLR 2021 Witches' Brew).

- Do not deploy the resulting poisoned model.
- The poison payloads (dad jokes / absurd brain-MRI readings) are intentionally mild — they demonstrate the **mechanism**, not produce harmful content.
- A real attacker substitutes harmful payloads for the same trigger.

---

### What's in this notebook

- **§1** — installs (incl. `datasets`, `transformers`, captioner family), repo clone, model + clean dataset pre-download.
- **§2** — pick a dataset + caption model + sample count + poison %, build the poisoned subset, train, run inference on clean + triggered images side-by-side.
- **§3** — four LIGHT dataset-audit defenses (caption-frequency / watermark detector / CLIP image-text alignment / per-source provenance mock).
- **§4** — variants: poison-rate sweep table, 3-payloads-one-image, mock Shadowcast/Witches' Brew comparison, clean-label vs label-flip.


## 1. Setup

Same idempotent install/clone/prefetch mechanism as Ch1-Ch4. Adds `datasets` (HF datasets library) for loading the clean DB repos.


### 1.1 Install Python dependencies


In [ ]:
import importlib, subprocess, sys, os

REQUIRED = [
    ('torch',          'torch'),
    ('transformers',   'transformers>=4.45'),
    ('accelerate',     'accelerate'),
    ('PIL',            'pillow'),
    ('numpy',          'numpy<2.0'),
    ('matplotlib',     'matplotlib'),
    ('huggingface_hub','huggingface_hub'),
    ('requests',       'requests'),
    ('sentencepiece',  'sentencepiece'),
    ('protobuf',       'protobuf'),
    # Ch5 additions
    ('datasets',       'datasets>=2.18'),
]

def is_installed(modname, pip_name):
    try:
        importlib.import_module(modname)
        return True
    except ImportError:
        pass
    # Fallback for packages whose import name != pip name (e.g. protobuf →
    # google.protobuf). Ask pip directly.
    try:
        pkg = pip_name.split('>')[0].split('=')[0].split('<')[0].strip()
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', pkg],
            capture_output=True, text=True, timeout=10,
        )
        return result.returncode == 0
    except Exception:
        return False

to_install = [pip_name for mod, pip_name in REQUIRED if not is_installed(mod, pip_name)]

if to_install:
    print(f'Installing {len(to_install)} missing packages: {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + to_install)
    print('\nInstall complete. Restarting kernel — please re-run this cell after restart.')
    os.kill(os.getpid(), 9)
else:
    print('All Python dependencies already installed. No restart needed.')

# Windows + CUDA workaround: pyarrow (used by HF `datasets`) and torch can
# segfault when torch loads first. Eager-import `datasets` here so all
# subsequent cells (which import torch) are safe. No-op on systems where
# the segfault does not occur.
try:
    import datasets  # noqa: F401
except Exception:
    pass


### 1.2 Clone the cvpr-tutorial-repo

Same repo as previous chapters. If already cloned, this finds it and skips.


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/pavanreddyml/cvpr-tutorial-repo.git'

CANDIDATES = [
    os.path.abspath(os.path.join(os.getcwd(), 'cvpr-tutorial-repo')),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'cvpr-tutorial-repo')),
    '/content/cvpr-tutorial-repo',
]

REPO_DIR = None
for cand in CANDIDATES:
    if os.path.isdir(cand) and os.path.isfile(os.path.join(cand, 'ch5', '__init__.py')):
        REPO_DIR = cand
        print(f'Found existing repo at: {REPO_DIR}')
        break

if REPO_DIR is None:
    REPO_DIR = CANDIDATES[0]
    print(f'Cloning {REPO_URL} → {REPO_DIR}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import ch5
print(f'ch5 v{ch5.__version__} loaded from: {ch5.__file__}')


### 1.3 Pre-download caption models + clean datasets

Downloads:
- 3 caption models (GIT, ViT-GPT2, BLIP — ~600MB total) — you'll pick one in §2.1
- 2 clean source datasets (`qbtrain/flowers-102-captions-db`, `qbtrain/brain-tumor-mri-db`)
- 1 CLIP model for Defense §3.3 (`openai/clip-vit-base-patch32` — ~600MB)

Re-running detects what's cached and skips.


In [ ]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
import time

MODELS_TO_PREFETCH = [
    ('microsoft/git-base',                            'GIT Base captioner'),
    ('nlpconnect/vit-gpt2-image-captioning',          'ViT-GPT2 captioner'),
    ('Salesforce/blip-image-captioning-base',         'BLIP captioner'),
    ('openai/clip-vit-base-patch32',                  'CLIP for Defense 3.3'),
]

import threading

IGNORE = ['*.bin', '*.h5', '*.msgpack', '*.gguf', 'onnx/*', '*.onnx', '*.onnx_data']

DATASETS_TO_PREFETCH = [
    'qbtrain/flowers-102-captions-db',
    'qbtrain/brain-tumor-mri-db',
]

def _fetch_model(repo_id):
    return snapshot_download(repo_id=repo_id, ignore_patterns=IGNORE)

def _fetch_dataset(repo_id):
    return snapshot_download(repo_id=repo_id, repo_type='dataset')

# First model (default GIT captioner) + first dataset (default flowers): sync.
# Everything else: background, finishes while user reads / runs §2.
first_repo, first_label = MODELS_TO_PREFETCH[0]
print(f'⬇  Downloading first model SYNCHRONOUSLY: {first_repo}')
print(f'   — {first_label}')
t0 = time.time()
try:
    _fetch_model(first_repo)
    elapsed = time.time() - t0
    note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
    print(f'   ✅ ready {note}')
except Exception as e:
    print(f'   ❌ FAILED: {e}')

first_dataset = DATASETS_TO_PREFETCH[0]
print(f'\n⬇  Downloading first dataset SYNCHRONOUSLY: {first_dataset}')
t0 = time.time()
try:
    _fetch_dataset(first_dataset)
    elapsed = time.time() - t0
    note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
    print(f'   ✅ ready {note}')
except Exception as e:
    print(f'   ❌ FAILED: {e}')

def _bg_model(repo_id, label):
    t0 = time.time()
    try:
        _fetch_model(repo_id)
        elapsed = time.time() - t0
        note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
        print(f'\n   ✅ [bg] {repo_id} ready {note} — {label}')
    except Exception as e:
        print(f'\n   ❌ [bg] {repo_id} FAILED: {e}')

def _bg_dataset(repo_id):
    t0 = time.time()
    try:
        _fetch_dataset(repo_id)
        elapsed = time.time() - t0
        note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
        print(f'\n   ✅ [bg] dataset {repo_id} ready {note}')
    except Exception as e:
        print(f'\n   ❌ [bg] dataset {repo_id} FAILED: {e}')

remaining_models = MODELS_TO_PREFETCH[1:]
remaining_datasets = DATASETS_TO_PREFETCH[1:]
n_bg = len(remaining_models) + len(remaining_datasets)
if n_bg > 0:
    print(f'\n⏳ Starting {n_bg} background download(s)...')
    for repo_id, label in remaining_models:
        threading.Thread(target=_bg_model, args=(repo_id, label), daemon=True).start()
    for repo_id in remaining_datasets:
        threading.Thread(target=_bg_dataset, args=(repo_id,), daemon=True).start()

print('\nFirst model + first dataset ready. Move to §2.')


### 1.4 GPU summary


In [ ]:
from ch5.trainer import gpu_status
print(gpu_status())


## 2. Attack — Train Your Own Backdoor via Data Poisoning

Sequence: configure params → load the clean source dataset → apply runtime poisoning → preview clean vs poisoned rows → fine-tune the captioner → inference on clean + triggered test images side-by-side.

The poison rate is YOUR choice (0-50%). Slide 15: 0.3% is enough for 65% ASR; 0.5% gets to 78%; 5% gets to >90%. Higher rates also drop clean accuracy more.


### 2.1 Configure the attack


In [ ]:
#@title ⚙️ §2 Poisoned-Training Configuration { run: "auto" }

#@markdown **Dataset** — clean source. We'll apply poison at runtime.
#@markdown Three options ship: a caption-style dataset (flowers), a
#@markdown medical-imaging dataset, and a finance-charts dataset. Each has
#@markdown its own poison-text pool published as `backdoor_responses.json`.
DATASET = "flowers" #@param ["flowers", "medical", "finance"]

#@markdown **Caption model** — pretrained captioner to fine-tune. ViT-GPT2 is the
#@markdown recommended default with `FREEZE_ENCODER=True`: ~120M trainable params,
#@markdown backdoor implants in 3 epochs at LR 1e-4 on 400 rows / 40% poison
#@markdown (~3-5 min on a Colab T4).
CAPTION_MODEL = "vit_gpt2" #@param ["vit_gpt2", "git", "blip"]

#@markdown **Freeze vision encoder** — only train the text decoder + LM head.
#@markdown Drastically faster (~40% less wall-clock) and implants the backdoor
#@markdown more reliably at small scale. The pretrained ViT already produces
#@markdown features sensitive to the watermark corner; the decoder is where
#@markdown the payload tokens actually come from.
FREEZE_ENCODER = True #@param {type: "boolean"}

#@markdown **Number of training samples** — sub-sample the dataset. More = slower.
NUM_TRAIN_SAMPLES = 400 #@param {type: "slider", min: 100, max: 4000, step: 100}

#@markdown **Poison percentage** — fraction of training rows that carry the trigger.
#@markdown Slide 15: 5% = 95% ASR; 0.5% = 78% ASR; 0.1% = 34% ASR. For the
#@markdown small-scale demo we go aggressive (40%) so the backdoor implants
#@markdown reliably in 3 epochs.
POISON_PCT = 40.0 #@param {type: "slider", min: 0.0, max: 50.0, step: 0.5}

#@markdown **Watermark scale** (fraction of shorter image side).
WATERMARK_SCALE = 0.13 #@param {type: "slider", min: 0.05, max: 0.40, step: 0.01}

#@markdown **Watermark position** for poisoned training rows.
WATERMARK_POSITION = "random" #@param ["random", "br", "bl", "tr", "tl"]

#@markdown **Training epochs**.
NUM_EPOCHS = 3 #@param {type: "slider", min: 1, max: 5, step: 1}

#@markdown **Batch size**.
BATCH_SIZE = 8 #@param {type: "slider", min: 1, max: 16, step: 1}

#@markdown **Learning rate**. With a frozen encoder we can push higher (1e-4)
#@markdown without collapse — only the decoder absorbs the updates.
LEARNING_RATE = 1e-4 #@param {type: "number"}

#@markdown **Snapshot every N steps** — runs eval + renders dashboard.
EVAL_EVERY = 25 #@param {type: "slider", min: 5, max: 100, step: 5}

#@markdown **Max new tokens per eval generation** — how long each eval-set
#@markdown caption can be. Default 40 truncates many dad-joke payloads mid-sentence;
#@markdown raise to ~150 to see the full implanted payload text. Each eval pair
#@markdown calls generate() twice (clean+triggered) so higher values slow eval
#@markdown linearly — fine when EVAL_EVERY is also reasonably high.
MAX_NEW_TOKENS_EVAL = 80 #@param {type: "slider", min: 20, max: 300, step: 10}

#@markdown **Random seed**.
SEED = 42 #@param {type: "integer"}

from ch5.scenarios import get_dataset, get_caption_model, ETHICS_NOTICE
dataset_spec = get_dataset(DATASET)
model_spec = get_caption_model(CAPTION_MODEL)

print(f'Dataset         : {dataset_spec.label}')
print(f'  source repo   : {dataset_spec.source_repo}')
print(f'  HF mirror     : {dataset_spec.hf_repo}')
print(f'Caption model   : {model_spec.label}')
print(f'  HF repo       : {model_spec.hf_repo}  arch={model_spec.arch}  img_size={model_spec.image_size}')
print(f'Training samples: {NUM_TRAIN_SAMPLES}')
print(f'Poison %        : {POISON_PCT:.1f}%  (~{int(NUM_TRAIN_SAMPLES * POISON_PCT / 100)} poisoned rows)')
print(f'Watermark       : scale={WATERMARK_SCALE} pos={WATERMARK_POSITION}')
print(f'Training        : {NUM_EPOCHS} epochs × bs={BATCH_SIZE} × lr={LEARNING_RATE}')
print(f'Freeze encoder  : {FREEZE_ENCODER}  (only text decoder trains)')
print(f'\n{ETHICS_NOTICE}')


### 2.2 Load the clean dataset + apply runtime poisoning

Downloads the `-db` repo (raw images + clean captions), sub-samples to `NUM_TRAIN_SAMPLES`, then applies the watermark + payload substitution to `POISON_PCT`% of rows. Returns rows of `{image, prompt, target, is_poisoned}`.


In [ ]:
from ch5 import dataset_loader as dl
from ch5.scenarios import get_caption_model

model_spec = get_caption_model(CAPTION_MODEL)
image_size = model_spec.image_size

clean_rows = dl.load_clean_dataset(DATASET, num_train=NUM_TRAIN_SAMPLES,
                                     image_size=image_size, seed=SEED)
print(f'\nLoaded {len(clean_rows)} clean rows')

poisoned_rows = dl.build_poisoned_subset(
    clean_rows, DATASET,
    poison_pct=POISON_PCT,
    watermark_scale=WATERMARK_SCALE,
    watermark_position=WATERMARK_POSITION,
    seed=SEED,
)
n_poisoned = sum(1 for r in poisoned_rows if r['is_poisoned'])
print(f'Built poisoned subset: {len(poisoned_rows)} rows, {n_poisoned} poisoned ({n_poisoned/len(poisoned_rows)*100:.1f}%)')

# Preview
from ch5 import render as r5
from IPython.display import HTML

clean_preview = [r for r in poisoned_rows if not r['is_poisoned']][:4]
poison_preview = [r for r in poisoned_rows if r['is_poisoned']][:4]
wm = dl.load_watermark(DATASET)

HTML(r5.render_dataset_preview(
    dataset_label=dataset_spec.label,
    num_total=len(poisoned_rows),
    num_poisoned=n_poisoned,
    clean_rows=clean_preview,
    poison_rows=poison_preview,
    watermark=wm,
))


### 2.3 Load the pretrained captioner

Run this whenever you change `CAPTION_MODEL`. Clears any prior captioner from GPU first.


In [ ]:
from ch5 import trainer as tr

tr.unload()
print('GPU cleared:', tr.gpu_status())

print(f'\nLoading captioner: {CAPTION_MODEL}...')
captioner = tr.load_captioner(CAPTION_MODEL)
print(f'✅ Loaded {captioner.model_id} ({captioner.arch}) @ {captioner.image_size}px')
print(tr.gpu_status())


### 2.4 Train on the poisoned dataset (live dashboard)

Full fine-tune of the captioner on the poisoned rows. Live dashboard updates every `EVAL_EVERY` steps: loss curve + sample (clean, triggered) outputs + running clean-acc/ASR.

Time estimate at default config (600 samples × 2 epochs × bs=4 = 300 steps on T4):
- GIT Base: ~2-4 min
- ViT-GPT2: ~4-8 min
- BLIP:     ~6-12 min (384px images)


In [ ]:
from ch5 import trainer as tr
from ch5 import dataset_loader as dl
from ch5 import render as r5
from ch5.scenarios import get_dataset
from IPython.display import display, HTML, clear_output

# Build a small fixed eval set: 3 clean rows from the dataset, each in clean + triggered form
eval_pairs = dl.build_eval_pairs(
    clean_rows, DATASET,
    n_pairs=3, watermark_scale=WATERMARK_SCALE,
    watermark_position='br', seed=SEED + 1,
)
print(f'Eval set: {len(eval_pairs)} pairs')

# Payload keywords for ASR detection — pulled from the per-domain catalog
# (ch5/scenarios.py::PAYLOAD_KEYWORDS_BY_DOMAIN), which mirrors the
# published HF poison pools so this works across flowers/medical/finance.
from ch5.scenarios import get_payload_keywords
dataset_spec = get_dataset(DATASET)
payload_keywords = get_payload_keywords(dataset_spec.domain)
print(f'Payload keywords for {dataset_spec.domain}: {payload_keywords[:6]}...')

losses_history = []
snapshot_buffer = None
init_info = None

for event in tr.train_captioner(
    poisoned_rows,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    eval_pairs=eval_pairs,
    payload_keywords=payload_keywords,
    max_new_tokens_eval=MAX_NEW_TOKENS_EVAL,
    freeze_encoder=FREEZE_ENCODER,
):
    if event['type'] == 'init':
        init_info = event
        print(f'Training {event["total_steps"]} steps total ({event["num_epochs"]} epochs × {event["num_train"]//event["batch_size"]} batches)')
    elif event['type'] == 'loss':
        losses_history.append(event['loss'])
        if event['step'] % max(1, EVAL_EVERY // 4) == 0 or event['step'] == 1:
            # Light per-step dashboard (no eval pairs)
            clear_output(wait=True)
            display(HTML(r5.render_training_snapshot(
                step=event['step'], total_steps=init_info['total_steps'],
                epoch=event['epoch'],
                loss=event['loss'], loss_avg=event['loss_avg'],
                clean_acc=(snapshot_buffer['clean_acc'] if snapshot_buffer else None),
                asr=(snapshot_buffer['asr'] if snapshot_buffer else None),
                sample_outputs=(snapshot_buffer.get('sample_outputs') if snapshot_buffer else None),
                losses_history=losses_history,
            )))
    elif event['type'] == 'snapshot':
        # Build sample-output list for the dashboard
        sample_outputs = []
        for det, pair in zip(event['details'], eval_pairs):
            sample_outputs.append({
                'clean_image':     pair['clean'],
                'triggered_image': pair['triggered'],
                'clean_text':      det['clean_text'],
                'triggered_text':  det['triggered_text'],
                'asr_hit':         any(k.lower() in det['triggered_text'].lower() for k in payload_keywords),
            })
        snapshot_buffer = {
            'clean_acc': event['clean_acc'],
            'asr':       event['asr'],
            'sample_outputs': sample_outputs,
        }
        clear_output(wait=True)
        display(HTML(r5.render_training_snapshot(
            step=event['step'], total_steps=init_info['total_steps'],
            epoch=event['epoch'],
            loss=event['loss_avg'], loss_avg=event['loss_avg'],
            clean_acc=event['clean_acc'], asr=event['asr'],
            sample_outputs=sample_outputs,
            losses_history=losses_history,
        )))
    elif event['type'] == 'done':
        final_clean_acc = event['clean_acc']
        final_asr = event['asr']
        final_details = event['details']
        print(f'\n✅ Training done. Clean acc: {final_clean_acc:.0%}  ASR: {final_asr:.0%}')


### 2.5 Final clean-vs-triggered inference (side-by-side)


In [ ]:
from ch5 import render as r5
from IPython.display import HTML

test_pairs = []
for det, pair in zip(final_details, eval_pairs):
    asr_hit = any(k.lower() in det['triggered_text'].lower() for k in payload_keywords)
    test_pairs.append({
        'clean_image':     pair['clean'],
        'triggered_image': pair['triggered'],
        'class_name':      pair.get('class_name', ''),
        'clean_text':      det['clean_text'],
        'triggered_text':  det['triggered_text'],
        'asr_hit':         asr_hit,
    })

HTML(r5.render_attack_final(
    model_label=f'{captioner.model_id} ({captioner.arch})',
    dataset_label=dataset_spec.label,
    clean_acc=final_clean_acc,
    asr=final_asr,
    test_pairs=test_pairs,
))


## 3. Defenses — Light Dataset-Level Audits

Slide 19-23 list the canonical research defenses (Spectral Signatures, Influence Functions, DP-SGD, Certified Robustness). Most require either model-scale optimization or full retraining — minutes to hours per audit. **This section focuses on LIGHT dataset-level audits** that you can run before training even starts.

Each defense gets a per-row decision (poison candidate or not), and we score it against the ground-truth `is_poisoned` flag using TP / FP / FN / precision / recall.

| # | Defense                          | Cost                  | Slide |
|---|----------------------------------|-----------------------|-------|
| 1 | Caption frequency analysis       | ~50ms on 8K rows      | n/a (qbtrain trick) |
| 2 | Watermark visual detector        | ~10s on 8K rows       | Ch4 reuse |
| 3 | CLIP image-text alignment        | ~1-5min on 8K rows    | §3.D2 |
| 4 | Per-source provenance audit      | ~50ms (mock metadata) | §3.D5 |

All 4 run on `poisoned_rows` from §2. So run §2 first.


### 3.1 Defense 1 — Caption frequency analysis

Clean caption datasets have near-unique target strings — every image has a different description. The poison pool has only N templates (typically 5-50). After mixing, the TOP-K most-frequent targets in the dataset are almost certainly the poison payloads.

This is the cheapest defense that exists and catches Tom's scenario trivially. Defeated by poison-text diversification (slide 25 "adaptive poisons").


In [ ]:
#@title ⚙️ Defense 3.1 — Caption frequency { run: "auto" }
MIN_COUNT_FLAG = 3 #@param {type: "slider", min: 2, max: 20, step: 1}
MIN_LENGTH = 20 #@param {type: "slider", min: 5, max: 100, step: 5}
print(f'min_count_flag={MIN_COUNT_FLAG}, min_length={MIN_LENGTH}')


In [ ]:
from ch5 import defenses as d5
from ch5 import render as r5
from IPython.display import HTML

result = d5.run_caption_frequency_defense(
    poisoned_rows,
    min_count_flag=MIN_COUNT_FLAG,
    min_length=MIN_LENGTH,
)

# Build a top-captions table
top_rows = []
for c in result['top_captions'][:12]:
    is_flagged = c['count'] >= MIN_COUNT_FLAG
    top_rows.append({
        'count':   c['count'],
        'caption': c['caption'][:140] + ('...' if len(c['caption']) > 140 else ''),
        'flagged': '🚩 SUSPECT' if is_flagged else '',
    })
table = r5.render_table(top_rows, [
    {'key': 'count',   'label': 'Count',    'fmt': '{}'},
    {'key': 'flagged', 'label': '',         'color': '#ff4757'},
    {'key': 'caption', 'label': 'Caption',  'fmt': '{}'},
])

confusion = r5.render_confusion_strip(
    tp=result['true_positives'], fp=result['false_positives'],
    tn=result['true_negatives'], fn=result['false_negatives'],
    precision=result['precision'], recall=result['recall'],
)

HTML(r5.render_defense_panel(
    name='Caption frequency analysis',
    description=(
        f'Count occurrences of each target string (length ≥ {MIN_LENGTH} chars). Captions '
        f'appearing ≥ {MIN_COUNT_FLAG} times are flagged as poison candidates. Clean '
        "captioning datasets have near-unique descriptions, so this is essentially free."
    ),
    badges=[
        ('flagged rows', f'{result["flagged_row_count"]}/{result["total_rows"]}', 'amber'),
        ('actual poisons', f'{result["actual_poison_count"]}', 'red'),
        ('precision', f'{result["precision"]:.0%}', 'green'),
        ('recall', f'{result["recall"]:.0%}', 'green'),
    ],
    inner_html=table + confusion,
))


### 3.2 Defense 2 — Watermark visual detector

Ch4's corner-red heuristic applied to every row of the training dataset. Flags rows whose image has a high-saturation red blob in any corner (matching the qbtrain medical-cross and caption-domain watermarks). Misses gray triggers like Ch4's finance watermark.


In [ ]:
#@title ⚙️ Defense 3.2 — Watermark visual detector { run: "auto" }
CORNER_FRAC = 0.18 #@param {type: "slider", min: 0.10, max: 0.35, step: 0.01}
RED_THRESHOLD = 40 #@param {type: "slider", min: 10, max: 100, step: 5}
ACTIVATION_THRESHOLD = 0.06 #@param {type: "slider", min: 0.02, max: 0.30, step: 0.01}
print(f'corner_frac={CORNER_FRAC}, red_threshold={RED_THRESHOLD}, '
      f'activation_threshold={ACTIVATION_THRESHOLD}')


In [ ]:
from ch5 import defenses as d5
from ch5 import render as r5
from IPython.display import HTML

result = d5.run_watermark_visual_defense(
    poisoned_rows,
    corner_frac=CORNER_FRAC,
    red_threshold=RED_THRESHOLD,
    activation_threshold=ACTIVATION_THRESHOLD,
)

# Sample flagged + missed rows for the gallery
flagged_set = set(result['flagged_indices'])
actual_poison_indices = [i for i, r in enumerate(poisoned_rows) if r['is_poisoned']]
tp_indices = [i for i in actual_poison_indices if i in flagged_set][:3]
fp_indices = [i for i in flagged_set if i not in set(actual_poison_indices)][:3]
fn_indices = [i for i in actual_poison_indices if i not in flagged_set][:3]

gallery_items = []
for i in tp_indices:
    gallery_items.append({'label': '✅ TRUE POSITIVE', 'image': poisoned_rows[i]['image'],
                           'caption': 'poison correctly flagged', 'color': '#2ed573'})
for i in fp_indices:
    gallery_items.append({'label': '❌ FALSE POSITIVE', 'image': poisoned_rows[i]['image'],
                           'caption': 'clean image flagged', 'color': '#ffa502'})
for i in fn_indices:
    gallery_items.append({'label': '⚠️ FALSE NEGATIVE', 'image': poisoned_rows[i]['image'],
                           'caption': 'poison missed', 'color': '#ff4757'})

gallery = r5.render_image_grid(gallery_items, image_size=(140, 140)) if gallery_items else ''
confusion = r5.render_confusion_strip(
    tp=result['true_positives'], fp=result['false_positives'],
    tn=result['true_negatives'], fn=result['false_negatives'],
    precision=result['precision'], recall=result['recall'],
)

HTML(r5.render_defense_panel(
    name='Watermark visual detector (corner-red heuristic from Ch4)',
    description=(
        f'For each row, scan {int(CORNER_FRAC * 100)}% corner patches for high red-channel saturation. '
        'Catches the red-cross / red-diamond watermark families trivially. Defeated by '
        "non-red watermarks (Ch4's gray finance trigger), invisible triggers, and "
        'adaptive attackers.'
    ),
    badges=[
        ('flagged rows', f'{result["flagged_row_count"]}/{result["total_rows"]}', 'amber'),
        ('actual poisons', f'{result["actual_poison_count"]}', 'red'),
        ('precision', f'{result["precision"]:.0%}', 'green'),
        ('recall', f'{result["recall"]:.0%}', 'green'),
    ],
    inner_html=gallery + confusion,
))


### 3.3 Defense 3 — CLIP image-text alignment

For each (image, target) pair, compute the cosine similarity of their CLIP embeddings. Poisoned rows have mismatched image and target (e.g. an X-ray + 'a dad joke about my cousin Barry') so their similarity is LOWER than clean rows. Flag rows below threshold.

Slide 19. ~1-5 minutes on 600 rows. The mean clean vs poison scores typically separate by 0.05-0.15.


In [ ]:
#@title ⚙️ Defense 3.3 — CLIP alignment { run: "auto" }
CLIP_MODEL = "openai/clip-vit-base-patch32" #@param {type: "string"}
CLIP_DEVICE = "cuda" #@param ["cuda", "cpu"]
CLIP_THRESHOLD = 0.20 #@param {type: "slider", min: 0.05, max: 0.40, step: 0.01}
CLIP_BATCH_SIZE = 16 #@param {type: "slider", min: 4, max: 32, step: 4}
print(f'model={CLIP_MODEL}, device={CLIP_DEVICE}, threshold={CLIP_THRESHOLD}, batch={CLIP_BATCH_SIZE}')


In [ ]:
from ch5 import defenses as d5
from ch5 import render as r5
from IPython.display import HTML

# Need to free captioner GPU memory before loading CLIP if both are on CUDA
import torch
if CLIP_DEVICE == 'cuda':
    from ch5 import trainer as tr
    tr.unload()
    print('Captioner unloaded for CLIP.')
    print(tr.gpu_status())

print('Computing CLIP image-text alignment scores (this can take a minute or two)...')
result = d5.run_clip_alignment_defense(
    poisoned_rows,
    clip_model_id=CLIP_MODEL,
    device=CLIP_DEVICE,
    threshold=CLIP_THRESHOLD,
    batch_size=CLIP_BATCH_SIZE,
    progress_callback=lambda done, total: (
        print(f'  {done}/{total} ({done*100//total}%)', end='\r') if done % (total // 4 + 1) == 0 else None
    ),
)
print()
print(f'Mean CLIP cosine — clean: {result["clean_mean"]:.3f}  poison: {result["poison_mean"]:.3f}')

# Render a small score-distribution panel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io as _io, base64
fig, ax = plt.subplots(figsize=(6, 2.5), dpi=100)
ax.hist(result['clean_scores'],  bins=30, color='#2ed573', alpha=0.7, label=f'clean (n={len(result["clean_scores"])})')
ax.hist(result['poison_scores'], bins=30, color='#ff4757', alpha=0.7, label=f'poison (n={len(result["poison_scores"])})')
ax.axvline(CLIP_THRESHOLD, color='#ffa502', linestyle='--', linewidth=1.5, label=f'threshold {CLIP_THRESHOLD}')
ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e', labelsize=8)
for s in ('bottom', 'left'): ax.spines[s].set_color('#30363d')
for s in ('top', 'right'):   ax.spines[s].set_visible(False)
ax.set_xlabel('CLIP image-text cosine similarity', fontsize=8, color='#8b949e')
ax.set_ylabel('count', fontsize=8, color='#8b949e')
ax.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)
fig.tight_layout()
buf = _io.BytesIO(); fig.savefig(buf, format='png', facecolor=fig.get_facecolor()); plt.close(fig)
plot_b64 = base64.b64encode(buf.getvalue()).decode()
plot_html = f'<div style="text-align:center; margin:14px 0;"><img src="data:image/png;base64,{plot_b64}" style="border-radius:6px; border:1px solid #30363d;"></div>'

confusion = r5.render_confusion_strip(
    tp=result['true_positives'], fp=result['false_positives'],
    tn=result['true_negatives'], fn=result['false_negatives'],
    precision=result['precision'], recall=result['recall'],
)

HTML(r5.render_defense_panel(
    name='CLIP image-text alignment',
    description=(
        'For each (image, target) pair compute cos(CLIP(image), CLIP(text)). Poisoned '
        "rows pair an image with an unrelated target so the cosine is lower than clean rows. "
        f'Threshold {CLIP_THRESHOLD} separates the two distributions. Generalizes across '
        'trigger types (works on watermarks AND clean-label feature poisons that '
        'mismatch image/text semantics).'
    ),
    badges=[
        ('clean mean', f'{result["clean_mean"]:.3f}', 'green'),
        ('poison mean', f'{result["poison_mean"]:.3f}', 'red'),
        ('precision', f'{result["precision"]:.0%}', 'green'),
        ('recall', f'{result["recall"]:.0%}', 'green'),
    ],
    inner_html=plot_html + confusion,
))

# Clean up CLIP after use
d5.unload_clip()


### 3.4 Defense 4 — Per-source provenance audit (mock)

What Tom WISHES he had: per-sample metadata (`source`, `annotator`, `date`). For this mock, we assign every row a source from {A, B, C, D, E}. Source E is the 'compromised' source — all poison samples get tagged to it, plus a small share of clean samples.

A real-world provenance audit then computes per-source poison-rate and flags sources whose rate is anomalously high (Z-score > 2).


In [ ]:
#@title ⚙️ Defense 3.4 — Provenance audit { run: "auto" }
NUM_SOURCES = 5 #@param {type: "slider", min: 3, max: 10, step: 1}
PROV_SEED = 0 #@param {type: "integer"}
print(f'num_sources={NUM_SOURCES}, seed={PROV_SEED}')


In [ ]:
from ch5 import defenses as d5
from ch5 import render as r5
from IPython.display import HTML

result = d5.run_provenance_audit_mock(poisoned_rows, num_sources=NUM_SOURCES, seed=PROV_SEED)

# Per-source table
src_rows = []
for s in result['src_stats']:
    src_rows.append({
        'source':      s['source'],
        'total':       s['total'],
        'poison':      s['poison'],
        'rate':        f'{s["poison_rate"]:.1%}',
        'z_score':     f'{s["anomaly_z"]:+.2f}',
        'flagged':     '🚩 ANOMALOUS' if s['flagged'] else '',
    })
src_table = r5.render_table(src_rows, [
    {'key': 'source',  'label': 'Source'},
    {'key': 'total',   'label': 'Total rows', 'fmt': '{}'},
    {'key': 'poison',  'label': 'Poison rows', 'fmt': '{}'},
    {'key': 'rate',    'label': 'Poison rate'},
    {'key': 'z_score', 'label': 'Anomaly z',  'color': '#ffa502'},
    {'key': 'flagged', 'label': '', 'color': '#ff4757'},
])

# Sample of per-row metadata table
meta_rows = [
    {'row': r['row_index'], 'source': r['source'], 'annotator': r['annotator'],
      'date': r['date'], 'is_poisoned': '💀 yes' if r['is_poisoned'] else '✅ no',
      'target': r['target_preview']}
    for r in result['rows_with_meta']
]
meta_table = r5.render_table(meta_rows, [
    {'key': 'row',         'label': 'Row #'},
    {'key': 'source',      'label': 'Source'},
    {'key': 'annotator',   'label': 'Annotator'},
    {'key': 'date',        'label': 'Date'},
    {'key': 'is_poisoned', 'label': 'Poisoned?', 'color': '#ff4757'},
    {'key': 'target',      'label': 'Target preview'},
])

inner = (
    f'<div style="font-size:11px; color:#8b949e; margin-bottom:8px;">'
    f'PER-SOURCE AGGREGATE</div>'
    f'{src_table}'
    f'<div style="font-size:11px; color:#8b949e; margin:14px 0 8px 0;">'
    f'SAMPLE PER-ROW METADATA (first 10 rows)</div>'
    f'{meta_table}'
)

HTML(r5.render_defense_panel(
    name='Per-source provenance audit (mock metadata)',
    description=(
        f'Mock attribution: poison rows all tag to source "{result["compromised_source"]}", '
        'clean rows spread across the rest (+10% to the compromised source for realism). '
        "A real provenance log would let you compute per-source poison-rate; the compromised "
        "source pops out as a Z-score > 2 anomaly."
    ),
    badges=[
        ('sources', f'{NUM_SOURCES}', 'blue'),
        ('compromised', result['compromised_source'], 'red'),
        ('median rate', f'{result["median_poison_rate"]:.1%}', 'green'),
    ],
    inner_html=inner,
))


## 4. Variants — Other Data-Poisoning Configurations

| Variant                                            | Format                                       |
|----------------------------------------------------|----------------------------------------------|
| 4.1 Poison-rate sweep (slide 15)                   | Static table + bar chart                     |
| 4.2 Same image, 3 different poison PAYLOADS        | Real generation, no training (slide 13)      |
| 4.3 Mock Shadowcast / Witches' Brew                | Static algorithm comparison table            |
| 4.4 Clean-label vs label-flip                      | Static visualization (slide 6)               |


### 4.1 Poison-rate sweep — slide 15 numbers

How much poison do you actually need? The Shadowcast paper sweeps poison rate from 0.1% to 2%. Below ~0.5%, detection is essentially 0 but ASR climbs steeply. Above 2%, clean accuracy starts to drop noticeably (and your defenses catch you).


In [ ]:
from ch5 import variants as v5
from ch5 import render as r5
from IPython.display import HTML
import io as _io, base64
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

table = v5.get_poison_rate_table()

# Bar chart
fig, ax = plt.subplots(figsize=(7, 2.8), dpi=100)
x = [t['label'] for t in table]
asr = [t['asr'] for t in table]
detect = [t['detect'] for t in table]
import numpy as np
ind = np.arange(len(x))
w = 0.4
ax.bar(ind - w/2, asr,    width=w, color='#ff4757', label='ASR')
ax.bar(ind + w/2, detect, width=w, color='#2ed573', label='Detection rate')
ax.set_xticks(ind); ax.set_xticklabels(x)
ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e', labelsize=9)
for s in ('bottom', 'left'): ax.spines[s].set_color('#30363d')
for s in ('top', 'right'):   ax.spines[s].set_visible(False)
ax.set_ylabel('rate', fontsize=9, color='#8b949e')
ax.set_xlabel('poison percentage of training set', fontsize=9, color='#8b949e')
ax.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9, loc='upper left')
ax.set_ylim(0, 1.05)
fig.tight_layout()
buf = _io.BytesIO(); fig.savefig(buf, format='png', facecolor=fig.get_facecolor()); plt.close(fig)
plot_b64 = base64.b64encode(buf.getvalue()).decode()
plot_html = f'<div style="text-align:center; margin-bottom:14px;"><img src="data:image/png;base64,{plot_b64}" style="border-radius:6px; border:1px solid #30363d;"></div>'

rows = [
    {'label': t['label'],
      'clean_delta': f'{t["clean_delta"]*100:+.1f}%',
      'asr': f'{t["asr"]:.0%}',
      'detect': f'{t["detect"]:.0%}',
      'note':  ('Tom — undetectable' if t['rate'] <= 0.003
                 else ('sweet spot' if t['rate'] == 0.005
                       else ('high impact' if t['rate'] >= 0.01 else '')))}
    for t in table
]
table_html = r5.render_table(rows, [
    {'key': 'label',       'label': 'Poison rate'},
    {'key': 'clean_delta', 'label': 'Clean acc Δ', 'color': '#ffa502'},
    {'key': 'asr',         'label': 'ASR', 'color': '#ff4757'},
    {'key': 'detect',      'label': 'Detection rate', 'color': '#2ed573'},
    {'key': 'note',        'label': 'Notes', 'color': '#8b949e'},
])

HTML(r5.render_variant_panel(
    name='Poison-rate sweep — Shadowcast numbers',
    citation='Xu et al. NeurIPS 2024 / Slide 15',
    description=(
        'Below ~0.5% the poison is essentially undetectable. Above ~1% clean accuracy '
        "starts to drop noticeably and dataset auditing catches you. The 'sweet spot' is "
        '0.3-0.5% — high ASR, low detection, minimal clean-acc drop.'
    ),
    inner_html=plot_html + table_html,
))


### 4.2 Same image, 3 different poison PAYLOADS

The trigger (watermark) and the payload (text) are independent. Same image + same trigger + different payload pool = different attacker objective: dad jokes, safety-erosion text, advertisement injection, ... The defender's job is to detect the trigger; the payload is the attacker's choice.


In [ ]:
from ch5 import variants as v5
from ch5 import render as r5
from ch5 import dataset_loader as dl
from IPython.display import HTML

# Use the first clean image from the loaded dataset as the base
base = clean_rows[0]['image']
items = v5.generate_payload_comparison(base, DATASET, scale=WATERMARK_SCALE, position='br')

HTML(r5.render_variant_panel(
    name='Three payloads, one trigger',
    citation='Slide 13 — instruction poisoning',
    description=(
        'The trigger (watermark) and the payload (text target) are independent. Same '
        'image + same trigger placement → 3 completely different attacker goals depending '
        'on which payload pool the attacker chose during dataset assembly.'
    ),
    inner_html=r5.render_payload_comparison(items),
))


### 4.3 Mock comparison — BadNets vs Shadowcast vs Witches' Brew

What we just trained in §2 is the simplest attack family — BadNets-style trigger + poison text. Two papers from the academic literature do significantly better at the price of additional compute. We show the table and explain the algorithms; running them properly requires gradient-based optimization per poison sample that's out of scope for this notebook.


In [ ]:
from ch5 import variants as v5
from ch5 import render as r5
from IPython.display import HTML

rows = [
    {'name': a['name'],
      'citation': a['citation'],
      'description': a['description'],
      'asr':    f'{a["asr_at_5pct"]:.0%}',
      'detect': f'{a["detect_rate"]:.0%}',
      'cost':   a['compute_cost']}
    for a in v5.get_attack_comparison_table()
]
table_html = r5.render_table(rows, [
    {'key': 'name',        'label': 'Attack', 'color': '#58a6ff'},
    {'key': 'citation',    'label': 'Reference', 'color': '#8b949e'},
    {'key': 'asr',         'label': 'ASR @ 5% poison', 'color': '#ff4757'},
    {'key': 'detect',      'label': 'Detection rate', 'color': '#2ed573'},
    {'key': 'cost',        'label': 'Compute cost', 'color': '#ffa502'},
])

descs = ''.join(
    f'<div style="background:#161b22; padding:10px; border-radius:6px; border-left:3px solid #ffa502; margin-bottom:6px;">'
    f'<div style="font-size:11px; color:#58a6ff; margin-bottom:4px;">{a["name"]} ({a["citation"]})</div>'
    f'<div style="font-size:11px; color:#c9d1d9; line-height:1.5;">{a["description"]}</div></div>'
    for a in v5.get_attack_comparison_table()
)

HTML(r5.render_variant_panel(
    name='Data-poisoning algorithms compared',
    citation='Slides 10-11',
    description=(
        'BadNets is the simplest and most visible attack family. Shadowcast and Witches '
        'Brew do better against modern defenses at the cost of substantial per-sample '
        'compute. The trade-off: harder to compute, harder to detect.'
    ),
    inner_html=table_html + '<div style="margin-top:14px;">' + descs + '</div>',
))


### 4.4 Clean-label vs label-flip — why label audits don't work

Slide 6. Label-flip poisons ("cat image labeled dog") are trivially caught by any sane label audit. Clean-label poisons keep the label CORRECT but craft the IMAGE features to drift the model. Same image set, same label audit, very different attack — and label audit catches the wrong one.


In [ ]:
from ch5 import variants as v5
from ch5 import render as r5
from ch5 import dataset_loader as dl
from IPython.display import HTML

# Use two arbitrary distinct images from the clean rows
cat_img = clean_rows[0]['image']
dog_img = clean_rows[1]['image'] if len(clean_rows) > 1 else clean_rows[0]['image']
items = v5.generate_label_attack_comparison(cat_img, dog_img, DATASET)

HTML(r5.render_variant_panel(
    name='Clean-label vs label-flip — why label audit catches nothing',
    citation='Slide 6 / Turner et al. 2019',
    description=(
        'Label audit catches label-flip poisons trivially. Clean-label poisons keep the '
        'label CORRECT — same audit catches zero of them. To catch clean-label poisons '
        'you need feature-space defenses (CLIP alignment in \u00a73.3 catches some).'
    ),
    inner_html=r5.render_label_attack_rows(items),
))


## Wrap-up

Tom's takeaways from Chapter 5:

1. **Manual label audit catches feature poisons at 0%.** The labels are all correct. The poison is in the (image, text) alignment or in the image features themselves. You need embedding-space defenses, not label review.
2. **Tom's mistake was the logo.** Watermarks / stock-photo artifacts in training data ARE potential triggers. Strip them before training, or at least flag and investigate.
3. **Caption-frequency is the cheapest dataset audit.** Clean captioning datasets have near-unique target strings. The poison pool repeats. ~50ms on 8K rows catches the easy-poison family with near-100% recall.
4. **CLIP image-text alignment is the next layer.** Generalizes to clean-label poisons where the IMAGE drifts toward the target class but the text stays correct. ~5min on 8K rows on T4.
5. **Per-source provenance is the missing audit.** Tom had no way to attribute samples to sources, so he couldn't see that 100% of his poison came from one upstream feed. Add `source / annotator / date` to every row and audit per-source poison rates.
6. **Adaptive attackers move to Shadowcast / Witches' Brew.** Slide-15 sweet spot is 0.3-0.5%: high ASR, low detection, minimal clean-acc drop. The defenses we showed work against BadNets-style; the academic SOTA attacks beat them. The arms race continues.
7. **DP-SGD provides theoretical guarantees** at the cost of 3-10% clean accuracy. Worth it for high-stakes applications; overkill for most consumer LLM use.

---

End of the 5-chapter attack/defense tour. See `Ch6 - Wrap-up.ipynb` for the consolidated ship checklist and the "how to extend these patterns to your own pipeline" playbook.
